# Silver Layer - Products

Transform raw Bronze data into clean Silver data.

**Source:** `end-to-end_pipeline.bronze.products`  
**Target:** `end-to-end_pipeline.silver.products`

**Approach:** Profile → Inspect → Transform → Validate

## Step 1: Profile Bronze Data

**Inspect data quality issues before transformation:**

* Duplicate product_ids
* NULL values in key fields (product_id, product_name, category, supplier_id)
* Invalid numeric values (list_price <= 0, standard_cost < 0)
* Inconsistent category values
* Inconsistent product_status values

This single query checks all quality dimensions.

In [0]:
%sql

SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT product_id) AS distinct_product_ids,
    COUNT(*) - COUNT(DISTINCT product_id) AS duplicate_products,

    SUM(CASE WHEN product_id IS NULL THEN 1 ELSE 0 END) AS null_product_ids,
    SUM(CASE WHEN product_name IS NULL THEN 1 ELSE 0 END) AS null_product_names,
    SUM(CASE WHEN category IS NULL THEN 1 ELSE 0 END) AS null_categories,
    SUM(CASE WHEN supplier_id IS NULL THEN 1 ELSE 0 END) AS null_supplier_ids,

    SUM(CASE WHEN list_price <= 0 THEN 1 ELSE 0 END) AS invalid_list_prices,
    SUM(CASE WHEN standard_cost < 0 THEN 1 ELSE 0 END) AS invalid_standard_costs,

    COUNT(DISTINCT category) AS category_variations,
    COUNT(DISTINCT product_status) AS product_status_variations

FROM `end-to-end_pipeline`.bronze.products;

total_rows,distinct_product_ids,duplicate_products,null_product_ids,null_product_names,null_categories,null_supplier_ids,invalid_list_prices,invalid_standard_costs,category_variations,product_status_variations
251,250,1,0,0,1,0,1,1,5,3


## Step 2: Inspect Categorical Values

**Review actual category and product_status values to identify standardization needs:**

* Category variations (" electronics " with spaces, null values)
* Product status casing (active → Active)
* Products with invalid prices or costs that will be filtered out

In [0]:
%sql

-- Inspect product categories
SELECT
    category,
    COUNT(*) AS records
FROM `end-to-end_pipeline`.bronze.products
GROUP BY category
ORDER BY records DESC;

category,records
Furniture,75
Accessories,68
Electronics,53
Office Supplies,53
electronics,1
null,1


In [0]:
%sql

-- Inspect product status values
SELECT
    product_status,
    COUNT(*) AS records
FROM `end-to-end_pipeline`.bronze.products
GROUP BY product_status
ORDER BY records DESC;

product_status,records
Active,224
Discontinued,26
active,1


In [0]:
%sql

-- Inspect products with invalid price or cost
SELECT
    product_id,
    product_name,
    list_price,
    standard_cost
FROM `end-to-end_pipeline`.bronze.products
WHERE list_price <= 0
   OR standard_cost < 0;

product_id,product_name,list_price,standard_cost
P0032,Nova Printer Supplies 32,0.0,584.29
P0046,Vertex Writing 46,550.33,-10.0


## Step 3: Transform to Silver

**Apply all data quality fixes in one pass:**

**Data Cleaning:**
* TRIM whitespace from all identifiers and text (handles " electronics " → "electronics")
* Standardize product names, categories, brands with INITCAP
* Convert NULL categories → 'Unknown'

**Categorical Standardization:**
* Standardize category casing (electronics → Electronics)
* Standardize product_status casing (active → Active)

**Data Type Enforcement:**
* Cast prices to DECIMAL(10,2)
* Convert launch_date to DATE format with TRY_TO_DATE

**Deduplication:**
* ROW_NUMBER() keeps first occurrence per product_id

**Data Quality Filters:**
* Remove NULL product_ids
* Remove products with list_price <= 0
* Remove products with standard_cost < 0

This creates a clean, analytics-ready Silver table.

In [0]:
%sql

CREATE OR REPLACE TABLE `end-to-end_pipeline`.silver.products AS

WITH cleaned AS (

    SELECT
        TRIM(product_id) AS product_id,

        INITCAP(TRIM(product_name)) AS product_name,

        CASE
            WHEN TRIM(category) IS NULL THEN 'Unknown'
            WHEN LOWER(TRIM(category)) = 'electronics' THEN 'Electronics'
            WHEN LOWER(TRIM(category)) = 'office supplies' THEN 'Office Supplies'
            WHEN LOWER(TRIM(category)) = 'furniture' THEN 'Furniture'
            WHEN LOWER(TRIM(category)) = 'accessories' THEN 'Accessories'
            ELSE INITCAP(TRIM(category))
        END AS category,

        INITCAP(TRIM(subcategory)) AS subcategory,

        INITCAP(TRIM(brand)) AS brand,

        TRIM(supplier_id) AS supplier_id,

        CAST(list_price AS DECIMAL(10,2)) AS list_price,

        CAST(standard_cost AS DECIMAL(10,2)) AS standard_cost,

        TRY_TO_DATE(launch_date, 'yyyy-MM-dd') AS launch_date,

        CASE
            WHEN LOWER(TRIM(product_status)) = 'active' THEN 'Active'
            WHEN LOWER(TRIM(product_status)) = 'discontinued' THEN 'Discontinued'
            ELSE INITCAP(TRIM(product_status))
        END AS product_status,

        ROW_NUMBER() OVER (
            PARTITION BY TRIM(product_id)
            ORDER BY product_id
        ) AS row_num

    FROM `end-to-end_pipeline`.bronze.products
)

SELECT
    product_id,
    product_name,
    category,
    subcategory,
    brand,
    supplier_id,
    list_price,
    standard_cost,
    launch_date,
    product_status

FROM cleaned

WHERE row_num = 1
  AND product_id IS NOT NULL
  AND list_price > 0
  AND standard_cost >= 0;

num_affected_rows,num_inserted_rows


## Step 4: Validate Silver Data

**Verify all transformations were successful.**

**Expected Results:**
* total_rows ≈ 248 (removed 3 invalid records from 251)
* distinct_product_ids = total_rows
* remaining_duplicates = 0
* null_product_ids = 0
* null_categories = 0
* invalid_prices = 0
* invalid_costs = 0
* invalid_product_status = 0 (only Active/Discontinued)
* invalid_categories = 0 (only Electronics/Office Supplies/Furniture/Accessories/Unknown)
* validation_status = 'PASS'

If any metric is unexpected, the transformation has an issue.

In [0]:
%sql

WITH validation AS (

    SELECT
        COUNT(*) AS total_rows,

        COUNT(DISTINCT product_id) AS distinct_product_ids,

        COUNT(*) - COUNT(DISTINCT product_id) AS remaining_duplicates,

        SUM(CASE WHEN product_id IS NULL THEN 1 ELSE 0 END)
            AS null_product_ids,

        SUM(CASE WHEN category IS NULL THEN 1 ELSE 0 END)
            AS null_categories,

        SUM(CASE WHEN list_price <= 0 THEN 1 ELSE 0 END)
            AS invalid_list_prices,

        SUM(CASE WHEN standard_cost < 0 THEN 1 ELSE 0 END)
            AS invalid_standard_costs,

        SUM(
            CASE
                WHEN product_status IN ('Active', 'Discontinued')
                THEN 0
                ELSE 1
            END
        ) AS invalid_product_status,

        SUM(
            CASE
                WHEN category IN (
                    'Electronics',
                    'Office Supplies',
                    'Furniture',
                    'Accessories',
                    'Unknown'
                )
                THEN 0
                ELSE 1
            END
        ) AS invalid_categories

    FROM `end-to-end_pipeline`.silver.products
)

SELECT
    *,

    CASE
        WHEN remaining_duplicates = 0
            AND null_product_ids = 0
            AND null_categories = 0
            AND invalid_list_prices = 0
            AND invalid_standard_costs = 0
            AND invalid_product_status = 0
            AND invalid_categories = 0
        THEN 'PASS'
        ELSE 'FAIL'
    END AS validation_status

FROM validation;

total_rows,distinct_product_ids,remaining_duplicates,null_product_ids,null_categories,invalid_list_prices,invalid_standard_costs,invalid_product_status,invalid_categories,validation_status
248,248,0,0,0,0,0,0,0,PASS
